# Small control workflow lab

This lab ties the pieces together: a controller starts, plans a route, reserves memory, waits for a heralding delay, and records a pair.


In [ ]:
from dataclasses import dataclass, field

from simyuj.components import PortKind
from simyuj.control import AgentContext, NodeAgent, SessionRuntime
from simyuj.control.payloads import AgentStart, TimerFired
from simyuj.engine import Timeline
from simyuj.entanglement import EntangledPairRecord, EntangledPairRegistry
from simyuj.network import Network, Node
from simyuj.resources import MemoryRef, MemorySlotState, ResourceManager


The controller does bookkeeping only. It uses a timer to model a delayed heralding result.


In [ ]:
def reserve_route_memory(controller, ctx: AgentContext) -> None:
    route = ctx.route_planner.fewest_hops_path(
        "alice",
        "bob",
        port_kind=PortKind.QUANTUM,
    )
    controller.route_links = route.link_ids

    reservation = ctx.resources.reserve_for_route(
        ctx.timeline.current_time,
        route,
        node_requirements=lambda node, index, length: 1 if index in (0, length - 1) else 0,
        reservation_id="reservation:alice-bob",
        created_at=ctx.timeline.current_time,
        expires_at=50,
        metadata=(("route", route.link_ids),),
    )
    committed = ctx.resources.commit(reservation.reservation_id)
    controller.reservation_id = committed.reservation_id
    controller.reservation_refs = committed.memory_ref_keys

    ctx.timers.set("herald-arrival", 7, correlation_id=committed.reservation_id)


In [ ]:
def record_heralded_pair(controller, timer: TimerFired, ctx: AgentContext) -> None:
    if timer.timer_id != "herald-arrival":
        return

    controller.timer_seen = timer.timer_id
    left = MemoryRef("alice", "mem", 0)
    right = MemoryRef("bob", "mem", 0)

    ctx.resources.mark_occupied(left)
    ctx.resources.mark_occupied(right)

    pair = ctx.pairs.register(
        EntangledPairRecord(
            pair_id="pair:alice-bob:0",
            left=left,
            right=right,
            fidelity=0.91,
            created_at=ctx.timeline.current_time,
            expires_at=ctx.timeline.current_time + 100,
            generation_link_id=controller.route_links[0],
            metadata=(("reservation", controller.reservation_id),),
        )
    )
    reserved = ctx.pairs.reserve(pair.pair_id)
    controller.pair_id = reserved.pair_id
    controller.pair_state = reserved.state.value

    for ref in (left, right):
        slot = ctx.resources.get_slot(ref)
        controller.resource_states.append((slot.ref.key, slot.state.value))


In [ ]:
@dataclass(slots=True)
class PairController(NodeAgent):
    route_links: tuple[str, ...] = ()
    reservation_id: str | None = None
    reservation_refs: tuple[tuple[str, str, int], ...] = ()
    timer_seen: str | None = None
    pair_id: str | None = None
    pair_state: str | None = None
    resource_states: list[tuple[tuple[str, str, int], str]] = field(default_factory=list)

    def on_start(self, start: AgentStart, ctx: AgentContext) -> None:
        reserve_route_memory(self, ctx)

    def on_timer(self, timer: TimerFired, ctx: AgentContext) -> None:
        record_heralded_pair(self, timer, ctx)


In [ ]:
timeline = Timeline(master_seed=101)
network = Network("control_workflow")

for node_id in ("controller", "alice", "bob"):
    network.add_node(Node(node_id))

network.add_quantum_link("q_alice_bob", "alice", "bob")

controller = PairController(agent_id="controller-agent", node_id="controller")
network.get_node("controller").add_agent(controller)

resources = ResourceManager()
resources.register_memory("alice", "mem", num_positions=1)
resources.register_memory("bob", "mem", num_positions=1)
pairs = EntangledPairRegistry()

runtime = SessionRuntime(
    timeline=timeline,
    network=network,
    resource_manager=resources,
    pair_registry=pairs,
    session_id="workflow-session",
)


Before the run, the ledgers are empty or free.


In [ ]:
print("Initial resources:")
for ref in resources.registered_memories():
    print(" ", ref.key, resources.get_slot(ref).state.value)

print("Initial pairs:", [pair.pair_id for pair in pairs.all_pairs()])


In [ ]:
runtime.run()

print("Timeline current time:", timeline.current_time)
print("Events scheduled:", timeline.events_scheduled)
print("Events executed:", timeline.events_executed)


The controller now has a route, a reservation, and a reserved pair record.


In [ ]:
print("Route links:", controller.route_links)
print("Reservation id:", controller.reservation_id)
print("Reservation refs:", controller.reservation_refs)
print("Timer seen:", controller.timer_seen)
print("Pair id:", controller.pair_id)
print("Pair state:", controller.pair_state)


In [ ]:
reservation = resources.get_reservation(controller.reservation_id)
pair = pairs.get(controller.pair_id)

print("Reservation owner:", reservation.owner)
print("Reservation state:", reservation.state.value)
print("Pair endpoints:", pair.memory_ref_keys)
print("Pair metadata:", pair.metadata)


In [ ]:
print("Resource ledger after heralding:")
for ref in resources.registered_memories():
    print(" ", ref.key, resources.get_slot(ref).state.value)

print("Pair registry:")
for item in pairs.all_pairs():
    print(" ", item.pair_id, item.state.value, item.fidelity)


Now finish the pair: consume it and mirror the memory slots as consumed.


In [ ]:
consumed_pair = pairs.consume(controller.pair_id)
for ref in consumed_pair.memory_refs:
    resources.mark_consumed(ref)

print("Consumed pair state:", consumed_pair.state.value)
print("Resource states after consume:")
for ref in resources.registered_memories():
    print(" ", ref.key, resources.get_slot(ref).state.value)


This is the control shape in miniature: agents decide, services schedule or update ledgers, and the timeline keeps ordering honest.
